# HAKE-MER — M1 ablation (H1 vs Step 0 baseline)

**Backbone:** DistilBERT-base · **Module:** M1 only (4 phrases × 32 tokens, additive attention)

**Protocol:** same as Step 0 — batch 16, LR 5e-5, 4 epochs, 3 seeds, best val F1-macro.

1. **Runtime → Factory reset** → **GPU**
2. Run all cells → download zip → archive in repo (see last cell)

Compare test F1-macro to Step 0 DistilBERT **0.486 ± 0.005**.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
f9df8aa


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.0 MB/s eta 0:00:00


In [4]:
!./run_m1_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 1.75MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 205kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 967kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 893kB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 24.6MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   2% 56.6k/2.77M [00:01<00:53, 50.4kB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:01<00:00, 2.17MB/s,  263kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:01<00:00, 2.20MB/s,  267kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:01<00:00, 318kB/s, 33.6kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:01<00:00, 322kB/s, 34.0kB/s  ]

simplified/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/347k [00

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
print("  epochs logged:", len(c["runs"][0]["history"]))
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_distilbert_base_uncased_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  epochs logged: 4
  f1_micro: 0.5635 ± 0.0019
  f1_macro: 0.4565 ± 0.0021
  exact_match: 0.4498 ± 0.0054
  map: 0.4907 ± 0.0040


In [6]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_{slug}_campaign.json")
out_name = "m1_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download m1_distilbert_step0.zip (3.4 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_m1/colab/`
2. Unzip + campaign JSON → `reference/artifacts/` (same pattern as Step 0)
3. Send zip to your dev machine or commit on GitHub